# Train a MuJoCo Playground environment for mjswan

One of MuJoCo Playground 1.1.0's RL environments, trained with Playground's own
environment and PPO settings, then exported as the ONNX policy its mjswan scene runs
([`train.py`](https://github.com/ttktjmt/mjswan_playground/blob/main/src/mjswan_playground/reaf/train.py)).

1. **Runtime → Change runtime type → T4 GPU**.
2. Add a Hugging Face **write** token as the Colab secret `HF_TOKEN` (the key icon on the
   left), with notebook access on.
3. Pick `ENV`, set `HF_USER` and **Runtime → Run all**. Every checkpoint is kept in the
   model repository on the Hub, one at least every 10 minutes, so when a session ends,
   **Run all** again resumes from the last one.
4. The result is in the same repository: `policy.onnx`, `contract.json` and
   `reference.npz`.

| `ENV` | Iterations | Worlds on a T4 |
|---|---|---|
| `CartpoleBalance`, `CartpoleSwingUp` | 200 | 4096 |
| `CartpoleBalanceVision` | 500 | 1024 |
| `SO101PickCube`, `TrossenPickCube` | 1000 | 4096 (Playground: 8192) |
| `SO101PickCubeVision`, `TrossenPickCubeVision` | 3000 | 4096 |
| `G1WholeBodyTracking` | 30000 | 4096 |

In [ ]:
!nvidia-smi

In [ ]:
import os
import re

from google.colab import userdata

ENV = "CartpoleBalance"  # @param ["CartpoleBalance", "CartpoleSwingUp", "CartpoleBalanceVision", "SO101PickCube", "SO101PickCubeVision", "TrossenPickCube", "TrossenPickCubeVision", "G1WholeBodyTracking"]
# The Hub account or organization the model repository is created under, public.
HF_USER = ""  # @param {type:"string"}
# 0: Playground's own, in the table above.
ITERATIONS = 0  # @param {type:"integer"}
# G1WholeBodyTracking's clip: 100STYLE's walk (CC BY 4.0), or a file in
# lvhaidong/LAFAN1_Retargeting_Dataset's g1/ without .csv (CC BY-NC-ND 4.0: not to be
# published).
MOTION = "100style_neutral_fw"  # @param {type:"string"}
# The mjswan_playground branch train.py is fetched from.
BRANCH = "main"  # @param {type:"string"}

if not HF_USER:
    raise ValueError("Set HF_USER: the Hub keeps what an ended session loses.")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
# One repository per environment (and clip): a run resumes from whatever it holds.
name = re.sub(r"(?<=[a-z0-9])(?=[A-Z])", "-", ENV).lower()
if ENV == "G1WholeBodyTracking":
    name += "-" + MOTION.replace("_", "-").lower()
HF_REPO = f"{HF_USER}/{name}"
OUT = f"/content/{ENV}"
RUN = f"--env {ENV} --out {OUT} --hf-repo {HF_REPO} --motion {MOTION}"
if ITERATIONS:
    RUN += f" --iterations {ITERATIONS}"
print("Model repository:", f"https://huggingface.co/{HF_REPO}")

In [ ]:
import urllib.request

urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/ttktjmt/mjswan_playground/refs/heads/"
    f"{BRANCH}/src/mjswan_playground/reaf/train.py",
    "train.py",
)
!python train.py setup
if _exit_code:  # noqa: F821, set by `!`
    raise RuntimeError("Setup failed: see above.")

Training curves while it runs (optional; this session's only).

In [ ]:
%load_ext tensorboard
%tensorboard --logdir "$OUT/checkpoints"

Train. If the session ends, **Run all** again: it resumes from the newest checkpoint in
`HF_REPO`.

In [ ]:
!python train.py train {RUN}
if _exit_code:  # noqa: F821, set by `!`
    raise RuntimeError("Training failed: see above. Nothing is exported.")

In [ ]:
!python train.py export {RUN}
if _exit_code:  # noqa: F821, set by `!`
    raise RuntimeError("The export failed: nothing is uploaded.")
!python train.py upload {RUN}